# Análisis Exploratorio de Datos - Hubway Bike Sharing

**Autor:** Guerra Chura Joan Leonardo

## Objetivo

Analizar el comportamiento del sistema Hubway mediante técnicas de Data Wrangling y Visual Analytics, identificando patrones temporales, espaciales, problemas de calidad y factores externos que pueden explicar la demanda.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",40)


# Paso 0: Metadata

El análisis utiliza dos tablas:

- `hubway_stations.csv`: información geográfica y administrativa de estaciones.
- `hubway_trips.csv`: registro individual de viajes.

Cada registro de trips representa un movimiento realizado por un usuario, con información temporal, espacial y características del usuario.

In [ ]:
stations = pd.read_csv("../data/hubway_stations.csv")
trips = pd.read_csv("../data/hubway_trips.csv", low_memory=False)

print(stations.shape)
print(trips.shape)


# Paso 1: Análisis del comportamiento de datos

Se evalúan:

- estructura del dataset,
- tipos de variables,
- duplicados,
- valores faltantes,
- consistencia temporal y espacial.

In [ ]:
trips.info()

print("Duplicados viajes:", trips.duplicated().sum())
print("Duplicados estaciones:", stations.duplicated().sum())


In [ ]:
trips['start_dt']=pd.to_datetime(trips['start_date'])
trips['end_dt']=pd.to_datetime(trips['end_date'])

missing=(trips.isna().sum()/len(trips)*100).sort_values(ascending=False)
missing


In [ ]:
plt.figure(figsize=(9,4))
missing[missing>0].plot(kind='bar')
plt.title("Porcentaje de valores faltantes por variable")
plt.ylabel("%")
plt.xticks(rotation=45)
plt.show()


## Análisis de nulos

Se verifica si los valores ausentes son aleatorios o si contienen información asociada al proceso de captura.

In [ ]:
pd.crosstab(trips['subsc_type'], trips['gender'].isna(), normalize='index')


# Paso 2: Análisis de Outliers

La duración del viaje puede contener:

- errores de captura,
- errores del sistema,
- eventos reales como bicicletas no devueltas durante largos periodos.

In [ ]:
q1=trips.duration.quantile(.25)
q3=trips.duration.quantile(.75)
iqr=q3-q1

upper=q3+1.5*iqr

print("Q1:",q1)
print("Q3:",q3)
print("Limite superior:",upper)
print("Duraciones negativas:",(trips.duration<0).sum())
print("Mayores a 24h:",(trips.duration>86400).sum())


In [ ]:
plt.figure(figsize=(9,3))
sns.boxplot(x=trips.loc[trips.duration<86400,'duration'])
plt.title("Boxplot de duración sin valores extremos")
plt.show()


# Paso 3: Visualización avanzada

Se incorporan gráficos temporales y espaciales para comprender el comportamiento del sistema.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=trips,x='subsc_type')
plt.title("Distribución por tipo de usuario")
plt.show()


In [ ]:
trips['hour']=trips.start_dt.dt.hour

plt.figure(figsize=(9,4))
trips.hour.value_counts().sort_index().plot(kind='bar')
plt.title("Demanda por hora del día")
plt.xlabel("Hora")
plt.ylabel("Viajes")
plt.show()


In [ ]:
trips['weekday']=trips.start_dt.dt.day_name()

heat=trips.pivot_table(
    index='weekday',
    columns='hour',
    values='hubway_id',
    aggfunc='count'
)

plt.figure(figsize=(12,5))
sns.heatmap(heat)
plt.title("Mapa de calor: día de semana vs hora")
plt.show()


In [ ]:
monthly=trips.groupby(trips.start_dt.dt.to_period('M')).size()

plt.figure(figsize=(12,4))
monthly.plot(marker='o')
plt.title("Evolución mensual de viajes")
plt.ylabel("Viajes")
plt.show()


In [ ]:
top_start=trips.strt_statn.value_counts().head(10)

plt.figure(figsize=(8,4))
top_start.plot(kind='bar')
plt.title("Top 10 estaciones de salida")
plt.xlabel("Estación")
plt.ylabel("Cantidad de viajes")
plt.show()


In [ ]:
station_usage=trips.strt_statn.value_counts()

map_data=stations.merge(
    station_usage.rename('demanda'),
    left_on='id',
    right_index=True,
    how='left'
)

plt.figure(figsize=(8,6))
plt.scatter(
    map_data.lng,
    map_data.lat,
    s=map_data.demanda.fillna(0)/50,
    alpha=0.5
)
plt.title("Distribución espacial de demanda por estación")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.show()


# Paso 4: Problema potencial encontrado

## Influencia del clima y localización espacial en la demanda

Pregunta:

**¿Cómo afectan las condiciones meteorológicas y la ubicación de las estaciones al número de viajes diarios?**

Se integra información climática externa mediante Open-Meteo.

In [ ]:
trips['fecha']=trips.start_dt.dt.date

daily=trips.groupby('fecha').size().reset_index(name='n_viajes')

params={
'latitude':42.355,
'longitude':-71.065,
'start_date':'2011-07-28',
'end_date':'2013-11-30',
'daily':'temperature_2m_max,precipitation_sum,snowfall_sum',
'timezone':'America/New_York'
}

response=requests.get(
'https://archive-api.open-meteo.com/v1/archive',
params=params
)

weather=pd.DataFrame(response.json()['daily'])
weather['time']=pd.to_datetime(weather['time']).dt.date

clima_viajes=daily.merge(
weather,
left_on='fecha',
right_on='time'
)

clima_viajes.head()


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))

sns.scatterplot(
data=clima_viajes,
x='temperature_2m_max',
y='n_viajes',
ax=ax[0]
)
ax[0].set_title("Temperatura vs viajes diarios")

sns.scatterplot(
data=clima_viajes,
x='precipitation_sum',
y='n_viajes',
ax=ax[1]
)
ax[1].set_title("Precipitación vs viajes diarios")

plt.show()


In [ ]:
print("Correlación temperatura:",
clima_viajes.n_viajes.corr(clima_viajes.temperature_2m_max))

print("Correlación precipitación:",
clima_viajes.n_viajes.corr(clima_viajes.precipitation_sum))


# Análisis adicional: comportamiento por clima y usuario

Se evalúa si usuarios registrados y ocasionales reaccionan de forma diferente ante condiciones meteorológicas adversas.

In [ ]:
clima_usuario=trips.groupby(
['fecha','subsc_type']
).size().reset_index(name='viajes')

clima_usuario.head()


# Conclusiones

El análisis identifica:

- problemas de calidad de datos,
- valores extremos,
- patrones horarios de movilidad,
- estaciones con mayor demanda,
- influencia potencial de factores ambientales.

La incorporación de variables climáticas y espaciales permite transformar un análisis descriptivo en un estudio espacio-temporal de demanda.